# 03 — Statistical / Analytical Validation
## Patient No-Show & Clinic Efficiency Analysis

**Objective:** formally test which patterns found in Notebook 02 are statistically real (not just noise from
sample-to-sample variation), and rank them by effect size so we know which ones matter most.

**Method:** for each categorical factor vs. `no_show_flag`, we run a **chi-square test of independence**
(is attendance actually related to this factor, or could the difference be due to chance?) and compute
**Cramér's V** (how strong is the relationship, on a 0-1 scale, independent of sample size).

**Important:** statistical significance (a small p-value) tells us a relationship is unlikely to be due to chance.
It does **not** tell us the relationship is causal, and with 110K+ rows, even tiny, practically meaningless
differences can become "statistically significant." Effect size (Cramér's V) is what tells us whether a
relationship is actually large enough to matter operationally — we lead with that, not with p-values alone.


In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/cleaned/cleaned_appointments.csv', parse_dates=['scheduled_day','appointment_day'])
print("Shape:", df.shape)


Shape: (110521, 24)


## 1. Chi-Square Test + Cramér's V Helper

Cramér's V ranges from 0 (no association) to 1 (perfect association). As a rough, widely used guide for a 2xN
table: **~0.1 = weak, ~0.3 = moderate, ~0.5+ = strong.** We'll apply this consistently to every factor below.


In [2]:
def test_association(df, factor_col, target_col='no_show_flag', min_group_size=30):
    """Chi-square test + Cramer's V for factor_col vs target_col.
    Drops categories with fewer than min_group_size rows first, and reports how many rows were dropped.
    """
    counts = df[factor_col].value_counts()
    small_groups = counts[counts < min_group_size].index.tolist()
    sub = df[~df[factor_col].isin(small_groups)].copy()
    dropped = len(df) - len(sub)

    contingency = pd.crosstab(sub[factor_col], sub[target_col])
    chi2, p, dof, expected = chi2_contingency(contingency)

    n = contingency.sum().sum()
    min_dim = min(contingency.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else np.nan

    return {
        'factor': factor_col,
        'n_used': n,
        'rows_dropped_small_groups': dropped,
        'small_groups_excluded': small_groups,
        'chi2': round(chi2, 2),
        'p_value': p,
        'cramers_v': round(cramers_v, 4)
    }

results = []


## 2. Testing Each Factor

We test the factors identified in the EDA as candidate drivers, plus a few more for completeness. Any category
with fewer than 30 records is excluded from that specific test (noted in the output) so a handful of rare cases
can't distort the result — consistent with the sample-size caution from Phase 4.


In [3]:
factors_to_test = ['age_group', 'gender', 'appointment_weekday', 'sms_received',
                   'waiting_time_group', 'hypertension', 'diabetes', 'alcoholism', 'scholarship']

for f in factors_to_test:
    res = test_association(df, f)
    results.append(res)
    print(f"{f:22s} | n={res['n_used']:>7,} | chi2={res['chi2']:>10.1f} | p={res['p_value']:.2e} | Cramer's V={res['cramers_v']:.4f}"
          + (f"  [excluded small groups: {res['small_groups_excluded']}]" if res['small_groups_excluded'] else ""))


age_group              | n=110,521 | chi2=     728.1 | p=4.05e-155 | Cramer's V=0.0812
gender                 | n=110,521 | chi2=       1.9 | p=1.72e-01 | Cramer's V=0.0041
appointment_weekday    | n=110,521 | chi2=      27.6 | p=4.37e-05 | Cramer's V=0.0158
sms_received           | n=110,521 | chi2=    1768.0 | p=0.00e+00 | Cramer's V=0.1265


waiting_time_group     | n=110,521 | chi2=    9624.0 | p=0.00e+00 | Cramer's V=0.2951
hypertension           | n=110,521 | chi2=     140.3 | p=2.25e-32 | Cramer's V=0.0356
diabetes               | n=110,521 | chi2=      25.2 | p=5.05e-07 | Cramer's V=0.0151


alcoholism             | n=110,521 | chi2=       0.0 | p=9.69e-01 | Cramer's V=0.0001
scholarship            | n=110,521 | chi2=      93.8 | p=3.54e-22 | Cramer's V=0.0291


### 2.1 Patient history (repeat patients) — tested separately

This factor needs its own setup since it depends on ordering each patient's appointments by date first
(same construction as Notebook 02, Section I).


In [4]:
df_sorted = df.sort_values(['patient_id', 'appointment_day']).copy()
df_sorted['prior_no_show'] = df_sorted.groupby('patient_id')['no_show_flag'].shift(1)
history_df = df_sorted.dropna(subset=['prior_no_show']).copy()
history_df['prior_no_show'] = history_df['prior_no_show'].astype(int)

res_history = test_association(history_df, 'prior_no_show', min_group_size=30)
results.append(res_history)
print(f"prior_no_show          | n={res_history['n_used']:>7,} | chi2={res_history['chi2']:>10.1f} | "
      f"p={res_history['p_value']:.2e} | Cramer's V={res_history['cramers_v']:.4f}")


prior_no_show          | n= 48,223 | chi2=    1370.3 | p=5.92e-300 | Cramer's V=0.1686


### 2.2 Neighbourhood — sample-size check first

81 neighbourhoods exist; several are very small. We report how many qualify at a reasonable threshold before
testing, and run the chi-square test only on the neighbourhoods with enough volume to be meaningful (using the
same >=100 threshold as the Excel and EDA phases, for consistency).


In [5]:
neigh_counts = df['neighbourhood'].value_counts()
print(f"Total neighbourhoods: {len(neigh_counts)}")
print(f"Neighbourhoods with < 30 appointments (excluded from any statistical test): {(neigh_counts < 30).sum()}")
print(f"Neighbourhoods with >= 100 appointments (used for the 'top no-show rate' ranking in Notebook 02): {(neigh_counts >= 100).sum()}")

res_neigh = test_association(df, 'neighbourhood', min_group_size=30)
results.append(res_neigh)
print(f"\nneighbourhood           | n={res_neigh['n_used']:>7,} | chi2={res_neigh['chi2']:>10.1f} | "
      f"p={res_neigh['p_value']:.2e} | Cramer's V={res_neigh['cramers_v']:.4f}")
print(f"(dropped {res_neigh['rows_dropped_small_groups']} rows from {len(res_neigh['small_groups_excluded'])} tiny neighbourhoods)")


Total neighbourhoods: 81
Neighbourhoods with < 30 appointments (excluded from any statistical test): 4
Neighbourhoods with >= 100 appointments (used for the 'top no-show rate' ranking in Notebook 02): 74



neighbourhood           | n=110,500 | chi2=     484.6 | p=8.50e-61 | Cramer's V=0.0662
(dropped 21 rows from 4 tiny neighbourhoods)


## 3. Ranked Summary — Strongest Observable Patterns

Sorted by Cramér's V (effect size), largest first. This is the ranking that should drive Phase 7 (segmentation)
and Phase 13 (final recommendations) — not p-value alone, since with 100K+ rows almost everything is
"statistically significant" even when the real-world effect is tiny.


In [6]:
summary = pd.DataFrame(results)[['factor','n_used','cramers_v','p_value','chi2']]
summary = summary.sort_values('cramers_v', ascending=False).reset_index(drop=True)
summary['p_value'] = summary['p_value'].apply(lambda x: f"{x:.2e}")
summary


,factor,n_used,cramers_v,p_value,chi2
0,waiting_time_group,110521,0.2951,0.00e+00,9624.01
1,prior_no_show,48223,0.1686,5.92e-300,1370.32
2,sms_received,110521,0.1265,0.00e+00,1767.98
3,age_group,110521,0.0812,4.05e-155,728.14
4,neighbourhood,110500,0.0662,8.50e-61,484.57
5,hypertension,110521,0.0356,2.25e-32,140.34
6,scholarship,110521,0.0291,3.54e-22,93.77
7,appointment_weekday,110521,0.0158,4.37e-05,27.59
8,diabetes,110521,0.0151,5.05e-07,25.25
9,gender,110521,0.0041,1.72e-01,1.87


## 4. Correlation Check (Numeric Factor)

`lead_time_days` is numeric/continuous rather than categorical, so we use point-biserial correlation
(equivalent to Pearson correlation against a 0/1 target) rather than chi-square.


In [7]:
corr = df[['lead_time_days','no_show_flag']].corr().iloc[0,1]
print(f"Correlation (lead_time_days vs no_show_flag): {corr:.4f}")
print("A correlation of ~0.19 is modest in absolute terms, but it's a real, monotonic relationship "
      "(confirmed by the bucketed rates in Notebook 02) rather than noise.")


Correlation (lead_time_days vs no_show_flag): 0.1863
A correlation of ~0.19 is modest in absolute terms, but it's a real, monotonic relationship (confirmed by the bucketed rates in Notebook 02) rather than noise.


## 6. Conclusion — What Carries Forward

The actual ranking by effect size (Cramer's V), from the executed results above:

| Rank | Factor | Cramer's V | Statistically significant? |
|---|---|---:|---|
| 1 | **Lead time (waiting_time_group)** | 0.295 | Yes |
| 2 | **Prior no-show (patient history)** | 0.169 | Yes |
| 3 | **SMS reminder status** | 0.127 | Yes |
| 4 | Age group | 0.081 | Yes |
| 5 | Neighbourhood | 0.066 | Yes |
| 6 | Hypertension | 0.036 | Yes |
| 7 | Scholarship (welfare enrollment) | 0.029 | Yes |
| 8 | Appointment weekday | 0.016 | Yes |
| 9 | Diabetes | 0.015 | Yes |
| 10 | Gender | 0.004 | **No** (p = 0.17) |
| 11 | Alcoholism | 0.0001 | **No** (p = 0.97) |

**This updates the read from Notebook 02.** Patient history looked like the single strongest pattern there
(largest raw percentage-point gap, 35.2% vs 18.0%), but once effect size is measured properly across the full
distribution, **lead time is actually the stronger driver overall** — patient history is a close second, and
notably it's measured on a smaller base (only the 48,223 appointments where a prior visit exists), while lead
time applies to the entire dataset. Both are legitimate top drivers and both should anchor Phase 7
(segmentation) and Phase 13 (final recommendations).

**Gender and alcoholism can be dropped as no-show drivers** — the data gives no statistical evidence they're
related to attendance at all, so building recommendations around them would not be defensible.

Everything from hypertension down to diabetes (ranks 4-9) is real but small — worth mentioning as minor,
secondary factors, not primary levers.